**Problem 10**

A study of length of hospital stay, in days, as a function of age, kind of health insurance, and whether or not the patient died while in the hospital. Length of hospital stay is recorded as a minimum of at least one day. The dataset is taken from:

UCLA Zero-Truncated Poisson Dataset

Fit the Zero-Truncated Poisson regression generalized linear model (GLM) to identify the factors associated with hospital length of stay. Interpret the results.

In [1]:
import pandas as pd
import numpy as np

import statsmodels.api as sm

from statsmodels.discrete.truncated_model import TruncatedLFPoisson

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error

In [5]:
pip install pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 27.6 MB/s eta 0:00:00


In [7]:
import pandas as pd
import pyreadstat
import urllib.request

In [8]:
url = "https://stats.idre.ucla.edu/stat/data/ztp.dta"

urllib.request.urlretrieve(
    url,
    "ztp.dta"
)

('ztp.dta', <http.client.HTTPMessage at 0x7b0e8dee7080>)

In [9]:
df, meta = pyreadstat.read_dta(
    "ztp.dta"
)

In [10]:
df.head()

,stay,age,hmo,died
0,4,4,0,0.0
1,9,4,1,0.0
2,3,7,1,1.0
3,9,6,0,0.0
4,1,7,0,1.0


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1493 entries, 0 to 1492
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   stay    1493 non-null   int64  
 1   age     1493 non-null   int64  
 2   hmo     1493 non-null   int64  
 3   died    1493 non-null   float64
dtypes: float64(1), int64(3)
memory usage: 46.8 KB


In [12]:
#Convert Variables
df['hmo'] = df['hmo'].astype('category')

df['died'] = df['died'].astype('category')

In [13]:
df.describe()

,stay,age
count,1493.000000,1493.000000
mean,9.728734,5.233758
std,8.132908,1.669273
min,1.000000,1.000000
25%,4.000000,4.000000
50%,8.000000,5.000000
75%,13.000000,6.000000
max,74.000000,9.000000


In [14]:
#Check Zero Counts
print(
    "Number of zeros:",
    (df['stay'] == 0).sum()
)

Number of zeros: 0


In [15]:
#Prepare Variables

#Response variable:

y = df['stay']

In [19]:
X = pd.get_dummies(
    df[['age', 'hmo', 'died']],
    drop_first=True
)

X = X.astype(float)

X = sm.add_constant(X)

In [20]:
#Fit Zero-Truncated Poisson Model

#Model:stay∼age+hmo+died
ztp_model = TruncatedLFPoisson(
    endog=y,
    exog=X
).fit()

Optimization terminated successfully.
         Current function value: 4.627461
         Iterations: 4
         Function evaluations: 6
         Gradient evaluations: 6


In [21]:
print(ztp_model.summary())

                    TruncatedLFPoisson Regression Results                     
Dep. Variable:                   stay   No. Observations:                 1493
Model:             TruncatedLFPoisson   Df Residuals:                     1489
Method:                           MLE   Df Model:                            3
Date:                Sun, 10 May 2026   Pseudo R-squ.:                 0.01294
Time:                        12:54:21   Log-Likelihood:                -6908.8
converged:                       True   LL-Null:                       -6999.4
Covariance Type:            nonrobust   LLR p-value:                 5.022e-39
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.4358      0.027     89.118      0.000       2.382       2.489
age           -0.0144      0.005     -2.868      0.004      -0.024      -0.005
hmo_1         -0.1359      0.024     -5.724      0.0

**Interpretation of Zero-Truncated Poisson Regression Results**

The Zero-Truncated Poisson regression model was used to identify factors associated with hospital length of stay, where zero-day stays are excluded from the dataset.

The overall model is statistically significant:

Likelihood Ratio p-value = 5.02 × 10⁻³⁹

This indicates that the predictors collectively help explain variation in hospital stay duration.

**Interpretation of Predictors**

**Age:**

Coefficient = -0.0144
p-value = 0.004

Age is statistically significant.

The negative coefficient suggests that higher age is associated with a slightly shorter expected hospital stay.

**Insurance Type (hmo_1)**

Coefficient = -0.1359
p-value < 0.001

Insurance type is statistically significant.

Patients with HMO insurance tend to have shorter hospital stays compared to the reference group.

**Mortality Status (died_1.0)**

Coefficient = -0.2038
p-value < 0.001

Mortality status is statistically significant.

Patients in the died = 1 category are associated with shorter expected hospital stays compared to the reference category.

**Interpretation of Coefficients**

The Zero-Truncated Poisson model uses a log link:

log(μ)=β0 +β1X1+⋯


Negative coefficients reduce the expected hospital stay duration.


**Overall Conclusion**

The analysis suggests that:

age,
insurance type,
and mortality status

are significant predictors of hospital length of stay. The Zero-Truncated Poisson model was appropriate because the dataset excludes zero-day hospital stays.